# PyOD with `cuml.accel` (GPU-backed scikit-learn)

Many PyOD detectors wrap **scikit-learn** estimators (for example `IsolationForest`, `PCA`, `NearestNeighbors`, `KMeans`). The RAPIDS library provides **`cuml.accel`**, which swaps supported scikit-learn estimators for **cuML** implementations on the GPU where available, with automatic CPU fallback otherwise.

**How it applies to PyOD**

- PyOD imports scikit-learn inside each model module (for example `pyod.models.iforest` does `from sklearn.ensemble import IsolationForest` at module load time), and that name is looked up again from the module's own globals every time `.fit()`/`.__init__()` runs.
- Because of that, **enabling `cuml.accel` after PyOD's modules are already imported does nothing by itself** -- `cuml.accel` patches `sys.modules["sklearn.xxx"]` in place, but the already-imported PyOD module still holds its own old, un-patched reference until you `importlib.reload()` it. Skipping this step is a *silent* failure: PyOD keeps working, just entirely on CPU, and a naive "CPU time vs GPU time" comparison will show ~1x speedup for every detector and looks like a benchmarking result rather than a bug. This notebook's own "Enable `cuml.accel`..." section below reloads every PyOD model module used above for exactly this reason -- do not skip it or re-order the cells.
- Alternatively, enable acceleration **before** any `sklearn` / PyOD import: `cuml.accel.install()`, the `CUML_ACCEL_ENABLED=1` environment variable, or `python -m cuml.accel your_script.py`. That avoids the reload step entirely, at the cost of not being able to show a CPU baseline first in the same session.

For each of five detectors (IForest, PCA, KNN, CBLOF, HDBSCAN), we time **`fit`** and **`predict`** on held-out test data **separately** with normal PyOD (CPU sklearn), **then** (after `cuml.accel.install()` and reload) repeat the same workflow.

**Requirements**

- NVIDIA GPU, CUDA, and a RAPIDS/cuML environment ([RAPIDS install](https://docs.rapids.ai/install)).
- Compatible `pyod` and scikit-learn versions ([cuml.accel limitations](https://docs.rapids.ai/api/cuml/stable/cuml-accel/limitations/)).

**Note:** Speedups depend on dataset size, GPU, and whether each estimator is fully accelerated or falls back to CPU. Small arrays can look slower on GPU because of transfer overhead. This demo uses 20K samples to keep runtime short; for meaningful GPU speedups, increase `N_TRAIN` to 100K+ (some algorithms like HDBSCAN may require significant CPU time at that scale).

**Known CPU-only detectors (verified against cuml 26.06 on an NVIDIA GB10, not just read off the docs):**

- `IForest` wraps `sklearn.ensemble.IsolationForest`, which is not in cuML's accelerated estimator list (only `RandomForestClassifier`/`RandomForestRegressor` are accelerated under `sklearn.ensemble`). Its "GPU" row is expected to match its CPU row.
- `HDBSCAN` (the PyOD detector) calls `sklearn.cluster.HDBSCAN.fit()` for the actual clustering, and a separate `sklearn.neighbors.NearestNeighbors` to score new points at predict time. Only the `NearestNeighbors` part is intercepted by `cuml.accel` -- `cuml.accel.profile()` never lists `HDBSCAN.fit` as a potentially-accelerated call at all, so the dominant cost (fitting) still runs on CPU. Its "GPU" fit time is expected to match its CPU fit time; only `predict` benefits.

`PCA`, `KNN` (`NearestNeighbors`), and `CBLOF` (`KMeans`) *are* accelerated end to end (confirmed both via `cuml.accel`'s own GPU-call logging and by checking `cuml.accel.is_proxy()` on each detector's core fitted estimator) and show real speedups once the reload step below has run.

## Setup: synthetic data and timing helper

In [1]:
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")

from pyod.utils.data import generate_data

RANDOM_STATE = 42
# N_TRAIN = 1_000_000
# N_TEST = 200_000
N_TRAIN = 20_000
N_TEST = 5_000
N_FEATURES = 32
CONTAMINATION = 0.1

X_train, X_test, y_train, y_test = generate_data(
    n_train=N_TRAIN,
    n_test=N_TEST,
    n_features=N_FEATURES,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    behaviour="new",
)
X_train = np.asarray(X_train, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

print("X_train:", X_train.shape, X_train.dtype, " X_test:", X_test.shape)

X_train: (20000, 32) float32  X_test: (5000, 32)


In [2]:
def _used_gpu_accel(detector) -> bool:
    """True if any fitted sub-estimator attribute is a cuml.accel GPU proxy.

    PyOD detectors stash their wrapped sklearn estimator under different
    attribute names (detector_, neigh_, clustering_estimator_, ...), so this
    checks all fitted attributes rather than hard-coding one name.
    """
    from cuml.accel import is_proxy

    return any(is_proxy(v) for v in vars(detector).values())


def time_fit_and_predict(detector, X_tr, X_te, require_gpu=False):
    """Wall times (seconds) for fit on train, then predict on test.

    If require_gpu=True, asserts that fit() actually ran through a
    cuml.accel GPU proxy. This exists to catch the case where cuml.accel
    was enabled *after* PyOD's model modules were already imported --
    PyOD then keeps calling the original CPU sklearn class silently, and
    without this check that looks like a (bad) benchmark result instead
    of a bug. See the "How it applies to PyOD" note above.
    """
    t0 = time.perf_counter()
    detector.fit(X_tr)
    t_fit = time.perf_counter() - t0
    if require_gpu:
        assert _used_gpu_accel(detector), (
            f"{type(detector).__name__}.fit did not execute any cuml.accel "
            "GPU calls -- did you run the 'Enable cuml.accel and reload PyOD "
            "model modules' cells below *before* this one, in a session "
            "where this detector's PyOD module had already been imported?"
        )
    t0 = time.perf_counter()
    detector.predict(X_te)
    t_predict = time.perf_counter() - t0
    return t_fit, t_predict


cpu_times = {}
gpu_times = {}

---
## 1. Isolation Forest (`IForest`)

Wraps `sklearn.ensemble.IsolationForest`.

In [3]:
from pyod.models.iforest import IForest

clf = IForest(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["IForest"] = {"fit": tf, "predict": tp}
print(f"IForest CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

IForest CPU (sklearn): fit=0.5963s  predict=0.0239s


---
## 2. PCA reconstruction-based detector (`PCA`)

Uses `sklearn.decomposition.PCA` internally.

In [4]:
from pyod.models.pca import PCA

clf = PCA(n_components=16, random_state=RANDOM_STATE)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["PCA"] = {"fit": tf, "predict": tp}
print(f"PCA CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

PCA CPU (sklearn): fit=0.0358s  predict=0.0008s


---
## 3. kNN distances (`KNN`)

Uses `sklearn.neighbors.NearestNeighbors` (and related neighbor search).

In [5]:
from pyod.models.knn import KNN

clf = KNN(n_neighbors=20, method="mean", metric="euclidean", n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["KNN"] = {"fit": tf, "predict": tp}
print(f"KNN CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

KNN CPU (sklearn): fit=0.2078s  predict=0.1255s


---
## 4. CBLOF with default `KMeans` (`CBLOF`)

Uses `sklearn.cluster.KMeans` for clustering by default.

In [6]:
from pyod.models.cblof import CBLOF

clf = CBLOF(n_clusters=16, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["CBLOF"] = {"fit": tf, "predict": tp}
print(f"CBLOF CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

CBLOF CPU (sklearn): fit=0.7807s  predict=0.0014s


---
## 5. HDBSCAN (`HDBSCAN`)

Uses `sklearn.cluster.HDBSCAN` for clustering and `sklearn.neighbors.NearestNeighbors` for scoring new points. Requires a recent scikit-learn that includes `sklearn.cluster.HDBSCAN`.

In [7]:
from pyod.models.hdbscan import HDBSCAN

clf = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric="euclidean",
    n_jobs=-1,
    contamination=CONTAMINATION,
)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["HDBSCAN"] = {"fit": tf, "predict": tp}
print(f"HDBSCAN CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

HDBSCAN CPU (sklearn): fit=9.0811s  predict=0.0864s


---
## Enable `cuml.accel` and reload PyOD model modules

Run the next **two** cells once (install, then reload), then run the GPU timing cells below.

The reload step is not optional here: `pyod.models.iforest`, `.pca`, `.knn`, `.cblof`, and `.hdbscan` were already imported by the CPU section above, each with its own `from sklearn.xxx import Yyy` bound at that time. `cuml.accel.install()` patches `sklearn`'s module cache going forward, but it cannot retroactively fix names other modules already bound before it ran. `importlib.reload()` re-executes those `from sklearn... import ...` lines against the now-patched `sklearn` modules, which is what actually lets the GPU-backed proxies take effect for PyOD's wrapped estimators.

In [8]:
# %load_ext cuml.accel  # equivalent IPython-magic form
import cuml

cuml.accel.install(log_level="debug")


[cuml.accel] Enabled managed memory.


[cuml.accel] Accelerator installed.


In [9]:
import importlib

import pyod.models.cblof
import pyod.models.hdbscan
import pyod.models.iforest
import pyod.models.knn
import pyod.models.pca

for _mod in (
    pyod.models.iforest,
    pyod.models.pca,
    pyod.models.knn,
    pyod.models.cblof,
    pyod.models.hdbscan,
):
    importlib.reload(_mod)

# Re-bind the detector classes in *this* notebook's namespace too, so `IForest`,
# `PCA`, etc. below point at the freshly reloaded module objects rather than the
# stale ones captured by the `from pyod.models... import ...` lines earlier.
from pyod.models.cblof import CBLOF
from pyod.models.hdbscan import HDBSCAN
from pyod.models.iforest import IForest
from pyod.models.knn import KNN
from pyod.models.pca import PCA

print("Reloaded PyOD model modules against the cuml.accel-patched sklearn.")

Reloaded PyOD model modules against the cuml.accel-patched sklearn.


---
## Same five workflows with `cuml.accel` enabled

Hyperparameters match the CPU section. Timings are **`fit`** on `X_train` and **`predict`** on `X_test` (binary labels; PyOD may call `decision_function` internally). `require_gpu=True` is passed for the three detectors verified (via `cuml.accel.profile()`, on cuml 26.06) to actually run their *core* estimator on GPU: `PCA`, `KNN`, `CBLOF`. `IForest` and `HDBSCAN` are left off the check -- see the notes at the top and by the `HDBSCAN` cell below for why.

In [10]:
# No require_gpu=True here: sklearn.ensemble.IsolationForest is not on cuML's
# accelerated estimator list, so this legitimately always runs on CPU -- a
# fit time close to the CPU row above is expected, not a regression.
clf = IForest(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
gpu_times["IForest"] = {"fit": tf, "predict": tp}
print(f"IForest GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

IForest GPU (cuml.accel): fit=0.6202s  predict=0.0240s


In [11]:
clf = PCA(n_components=16, random_state=RANDOM_STATE)
tf, tp = time_fit_and_predict(clf, X_train, X_test, require_gpu=True)
gpu_times["PCA"] = {"fit": tf, "predict": tp}
print(f"PCA GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

[cuml.accel] `PCA.fit` ran on GPU


[cuml.accel] `PCA` fitted attributes synced to CPU


PCA GPU (cuml.accel): fit=0.3408s  predict=0.0012s


In [12]:
clf = KNN(n_neighbors=20, method="mean", metric="euclidean", n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test, require_gpu=True)
gpu_times["KNN"] = {"fit": tf, "predict": tp}
print(f"KNN GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

[cuml.accel] `NearestNeighbors.fit` ran on GPU


[cuml.accel] `NearestNeighbors` fitted attributes synced to CPU


[cuml.accel] `NearestNeighbors.kneighbors` ran on GPU


[cuml.accel] `NearestNeighbors.kneighbors` ran on GPU


KNN GPU (cuml.accel): fit=0.1584s  predict=0.0198s


In [13]:
clf = CBLOF(n_clusters=16, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test, require_gpu=True)
gpu_times["CBLOF"] = {"fit": tf, "predict": tp}
print(f"CBLOF GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

[cuml.accel] `KMeans.fit` ran on GPU


[cuml.accel] `KMeans` fitted attributes synced to CPU


[cuml.accel] `KMeans.predict` ran on GPU


CBLOF GPU (cuml.accel): fit=0.1666s  predict=0.0026s


In [14]:
clf = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric="euclidean",
    n_jobs=-1,
    contamination=CONTAMINATION,
)
# No require_gpu=True here: as of cuml 26.06, cuml.accel.profile() shows only
# the *auxiliary* NearestNeighbors call (used to score new points) is
# intercepted for this detector -- the dominant cost, the core
# sklearn.cluster.HDBSCAN.fit() clustering itself, never shows up in the
# profiler's list of potentially-accelerated calls and always runs on CPU.
# Asserting require_gpu=True here would still pass (vars(clf) also contains
# the accelerated .tree_ NearestNeighbors), which would hide that mismatch
# rather than catch it -- exactly the kind of misleading "success" this
# notebook is trying to avoid.
tf, tp = time_fit_and_predict(clf, X_train, X_test)
gpu_times["HDBSCAN"] = {"fit": tf, "predict": tp}
print(f"HDBSCAN GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

[cuml.accel] `NearestNeighbors.fit` ran on GPU


[cuml.accel] `NearestNeighbors.kneighbors` ran on GPU


HDBSCAN GPU (cuml.accel): fit=10.0339s  predict=0.0058s


## Summary table

In [15]:
rows = []
for name in ("IForest", "PCA", "KNN", "CBLOF", "HDBSCAN"):
    cf = cpu_times[name]["fit"]
    cp = cpu_times[name]["predict"]
    gf = gpu_times[name]["fit"]
    gp = gpu_times[name]["predict"]
    sf = cf / gf if gf > 0 else float("nan")
    sp = cp / gp if gp > 0 else float("nan")
    rows.append((name, cf, cp, gf, gp, sf, sp))

try:
    import pandas as pd

    display(
        pd.DataFrame(
            rows,
            columns=[
                "detector",
                "cpu_fit_s",
                "cpu_predict_s",
                "gpu_fit_s",
                "gpu_predict_s",
                "fit_speedup_cpu/gpu",
                "predict_speedup_cpu/gpu",
            ],
        )
    )
except Exception:
    hdr = (
        f"{'detector':<10} {'cpu_fit':>10} {'cpu_pred':>10} "
        f"{'gpu_fit':>10} {'gpu_pred':>10} {'fit_sp':>8} {'pred_sp':>8}"
    )
    print(hdr)
    for name, cf, cp, gf, gp, sf, sp in rows:
        print(
            f"{name:<10} {cf:10.4f} {cp:10.4f} {gf:10.4f} {gp:10.4f} "
            f"{sf:8.2f}x {sp:8.2f}x"
        )

,detector,cpu_fit_s,cpu_predict_s,gpu_fit_s,gpu_predict_s,fit_speedup_cpu/gpu,predict_speedup_cpu/gpu
0,IForest,0.596350,0.023859,0.620199,0.024035,0.961546,0.992673
1,PCA,0.035793,0.000847,0.340833,0.001246,0.105018,0.679955
2,KNN,0.207837,0.125537,0.158388,0.019752,1.312197,6.355562
3,CBLOF,0.780693,0.001448,0.166551,0.002627,4.687425,0.551300
4,HDBSCAN,9.081130,0.086400,10.033872,0.005832,0.905047,14.815217


## Optional: other ways to enable acceleration

- **Programmatic (before sklearn / pyod):** `cuml.accel.install()`
- **CLI:** `python -m cuml.accel your_script.py`
- **Environment:** `export CUML_ACCEL_ENABLED=1`

See [logging and profiling](https://docs.rapids.ai/api/cuml/stable/cuml-accel/logging-and-profiling/) to see GPU vs CPU fallback per call.